### 1174. Immediate Food Delivery II
```
Table: Delivery

+-----------------------------+---------+
| Column Name                 | Type    |
+-----------------------------+---------+
| delivery_id                 | int     |
| customer_id                 | int     |
| order_date                  | date    |
| customer_pref_delivery_date | date    |
+-----------------------------+---------+
delivery_id is the column of unique values of this table.
The table holds information about food delivery to customers that make orders at some date and specify a preferred delivery date (on the same order date or after it).
 

If the customer's preferred delivery date is the same as the order date, then the order is called immediate; otherwise, it is called scheduled.

The first order of a customer is the order with the earliest order date that the customer made. It is guaranteed that a customer has precisely one first order.

Write a solution to find the percentage of immediate orders in the first orders of all customers, rounded to 2 decimal places.

The result format is in the following example.

 

Example 1:

Input: 
Delivery table:
+-------------+-------------+------------+-----------------------------+
| delivery_id | customer_id | order_date | customer_pref_delivery_date |
+-------------+-------------+------------+-----------------------------+
| 1           | 1           | 2019-08-01 | 2019-08-02                  |
| 2           | 2           | 2019-08-02 | 2019-08-02                  |
| 3           | 1           | 2019-08-11 | 2019-08-12                  |
| 4           | 3           | 2019-08-24 | 2019-08-24                  |
| 5           | 3           | 2019-08-21 | 2019-08-22                  |
| 6           | 2           | 2019-08-11 | 2019-08-13                  |
| 7           | 4           | 2019-08-09 | 2019-08-09                  |
+-------------+-------------+------------+-----------------------------+
Output: 
+----------------------+
| immediate_percentage |
+----------------------+
| 50.00                |
+----------------------+
Explanation: 
The customer id 1 has a first order with delivery id 1 and it is scheduled.
The customer id 2 has a first order with delivery id 2 and it is immediate.
The customer id 3 has a first order with delivery id 5 and it is scheduled.
The customer id 4 has a first order with delivery id 7 and it is immediate.
Hence, half the customers have immediate first orders.

```

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql.functions import col,when,count,sum,round,min

In [3]:
spark=SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [4]:
Data = [[1, 1, '2019-08-01', '2019-08-02'], [2, 2, '2019-08-02', '2019-08-02'], [3, 1, '2019-08-11', '2019-08-12'],
        [4, 3, '2019-08-24', '2019-08-24'], [5, 3, '2019-08-21', '2019-08-22'], [6, 2, '2019-08-11', '2019-08-13'],
        [7, 4, '2019-08-09', '2019-08-09']]
Schema=StructType([StructField("delivery_id",IntegerType(),True),\
                  StructField("customer_id",IntegerType(),True),\
                   StructField("order_date",StringType(),True),\
                   StructField("customer_pref_delivery_date",StringType(),True)]
                  
                 )
Delivery=spark.createDataFrame(Data,Schema)
Delivery=Delivery.withColumn("order_date",col("order_date").cast("date")).withColumn("customer_pref_delivery_date",col("customer_pref_delivery_date").cast("date"))

In [5]:
Delivery.show()

+-----------+-----------+----------+---------------------------+
|delivery_id|customer_id|order_date|customer_pref_delivery_date|
+-----------+-----------+----------+---------------------------+
|          1|          1|2019-08-01|                 2019-08-02|
|          2|          2|2019-08-02|                 2019-08-02|
|          3|          1|2019-08-11|                 2019-08-12|
|          4|          3|2019-08-24|                 2019-08-24|
|          5|          3|2019-08-21|                 2019-08-22|
|          6|          2|2019-08-11|                 2019-08-13|
|          7|          4|2019-08-09|                 2019-08-09|
+-----------+-----------+----------+---------------------------+



In [6]:
delivery_min=Delivery.groupBy("customer_id").agg(min("order_date").alias("min_oder_date"))
delivery_min.show()

+-----------+-------------+
|customer_id|min_oder_date|
+-----------+-------------+
|          1|   2019-08-01|
|          2|   2019-08-02|
|          3|   2019-08-21|
|          4|   2019-08-09|
+-----------+-------------+



In [7]:
D=Delivery.select("customer_id","order_date","customer_pref_delivery_date").alias("D")
DM=delivery_min.select("*").alias("DM")

In [9]:
DL= D.join(DM,on=[(col("D.customer_id")==col("DM.customer_id")) & (col("D.order_date")==col("DM.min_oder_date"))],how="inner")

In [10]:
result= (DL.agg(round(
    100*(sum(when(
        col("order_date")==col("customer_pref_delivery_date"),1).otherwise(0)
    ) / count("*")) ,2).alias("immediate_percentage")
    
)
        )

In [11]:
result.show()

+--------------------+
|immediate_percentage|
+--------------------+
|                50.0|
+--------------------+



In [12]:
spark.stop()